# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema, accessible via URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the Croissant metadata and explore core attributes and documentation with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()
print("Dataset Title: ", metadata.get("name", "<no name>"))
print("Description: ", metadata.get("description", "<no description>"))
print("Version: ", metadata.get("version", ""))
print("License: ", metadata.get("license", ""))

# For reference, show publication date, identifier, and list of keywords if present
print("Published: ", metadata.get("datePublished", "N/A"))
print("Identifier: ", metadata.get("identifier", ""))
if "keywords" in metadata:
    print("Keywords:", ", ".join(str(k) for k in metadata["keywords"]))

## 2. Data Overview

Review available record sets, their fields, and corresponding Croissant IDs (`@id`). All exploration will reference entities by their `@id` for reproducibility and interoperability.

Let's list all record sets and their fields present in the dataset.

In [ ]:
# List all available record sets with their @id and field IDs
record_sets = dataset.metadata.get("recordSet", [])

if not record_sets:
    print("No record sets found in the dataset's top-level metadata. Attempting to scan for record sets declared elsewhere.")
    # fallback: record sets can sometimes be found as top-level objects with '@type': 'RecordSet'
    # scan for record set objects in all metadata (flattened)
    def find_record_sets(obj):
        found = []
        if isinstance(obj, dict):
            if obj.get("@type", "") == "RecordSet":
                found.append(obj)
            for v in obj.values():
                found.extend(find_record_sets(v))
        elif isinstance(obj, list):
            for v in obj:
                found.extend(find_record_sets(v))
        return found
    all_record_obj = find_record_sets(metadata)
    print(f"Found {len(all_record_obj)} record set(s):\n")
    for rec in all_record_obj:
        print(f"- RecordSet name: {rec.get('name', '<no name>')} | @id: {rec.get('@id')}\n  Description: {rec.get('description', '<no desc>')}")
        fields = rec.get("field", [])
        if fields:
            print("  Fields:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"   * {f.get('name', f.get('@id'))} (id: {f.get('@id', f)})")
                else:
                    print(f"   * {f}")
        print()
else:
    print("Record Sets listed as recordSet property:")
    for rec in record_sets:
        rec_obj = rec if isinstance(rec, dict) else {'@id': rec}
        print(f"- {rec_obj.get('@id')}")

Let us examine one record (if possible) from each top-level record set for an idea of the schema and columns.

In [ ]:
# List all recordSet @ids for use. We'll scan the metadata again to get actual @ids for further extraction.
def get_record_set_ids(obj):
    ids = []
    if isinstance(obj, dict):
        if obj.get("@type", "") == "RecordSet" and "@id" in obj:
            ids.append(obj["@id"])
        for v in obj.values():
            ids.extend(get_record_set_ids(v))
    elif isinstance(obj, list):
        for v in obj:
            ids.extend(get_record_set_ids(v))
    return ids

record_set_ids = get_record_set_ids(metadata)
if record_set_ids:
    print("RecordSet @ids detected:")
    for i, rid in enumerate(record_set_ids):
        print(f"{i+1}. {rid}")
else:
    print("No record sets detected. Unable to continue record inspection.")

# Show the first 1-2 records for each detected record set
for rsid in record_set_ids:
    print(f"\nFirst records for record set: {rsid}")
    try:
        iterator = dataset.records(record_set=rsid)
        for idx, record in enumerate(iterator):
            print(json.dumps(record, indent=2))
            if idx > 0:
                break
    except Exception as e:
        print("Could not extract records due to error:", e)


## 3. Data Extraction

Extract tabular data from record sets into pandas DataFrames. We reference all record sets by their `@id`.

> **Tip**: Choose the record set representing the main protein quantitation table for detailed EDA. Adjust the `main_recordset_id` as needed.

In [ ]:
# Set up all available record set ids for batch extraction
dataframes = {}

# For this dataset, the main data is usually in the first or main protein quantitation record set.
record_sets_to_load = record_set_ids[:]

for rsid in record_sets_to_load:
    print(f"Attempting to extract record set: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records; Columns: {df.columns.tolist()}")
        else:
            print("No records found in set.")
    except Exception as e:
        print(f"Error loading records from {rsid}: {e}")

# Choose a main record set for further EDA (pick the largest if in doubt)
if dataframes:
    # Pick the DataFrame with most columns and records
    main_recordset_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nUsing main record set: {main_recordset_id}")
    print("Columns:", dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())
else:
    print("No extractable table record sets available.")

## 4. Exploratory Data Analysis (EDA)

Demonstrate common EDA operations: filtering, normalization, outlier removal, and grouping by biological attributes. We use only attribute `@id`/column names found in the extracted DataFrame.

In [ ]:
# Select a numeric field for analysis (choose from DataFrame columns, e.g., MW or coverage)
df = dataframes[main_recordset_id]
print("Columns available for selection (use their @id or column name):", df.columns.tolist())

# Example choice (typical column IDs): 'MW' (molecular weight), 'Coverage', 'PeptideCount', etc.
numeric_field = None
# Try to autodetect common numeric fields
for c in df.columns:
    if c.lower().startswith("mw"):
        numeric_field = c
        break
if numeric_field is None:
    for c in df.columns:
        if "coverage" in c.lower() or "peptide" in c.lower():
            numeric_field = c
            break
if numeric_field is None:
    numeric_field = df.columns[df.dtypes == 'float64'][0] if any(df.dtypes == 'float64') else df.columns[0]

print(f"Using numeric field for EDA: {numeric_field}")

# Coerce values to numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Outlier filtering: threshold = mean + 2*std as example
threshold = df[numeric_field].mean() + 2*df[numeric_field].std()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {round(threshold,2)} (outlier high values):")
display(filtered_df.head())

# Normalizing the numeric field
df[f"{numeric_field}_normalized"] = (df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
print(f"Normalized {numeric_field} (z-score example, entire set):")
display(df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a main identifier, for instance 'Accession' or 'Sample' if present
group_field = None
for f in ['Accession', 'Sample', 'accession', 'sample']:
    if f in df.columns:
        group_field = f
        break
if group_field:
    grouped_df = df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field}, mean of {numeric_field}:")
    display(grouped_df.head())
else:
    print("No suitable grouping field ('Sample', 'Accession') found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to key categorical/grouping fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (normalized and raw)
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='steelblue')
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.tight_layout()
plt.show()

# If normalized column exists, plot it
if f"{numeric_field}_normalized" in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30, color='darkorange')
    plt.title(f"Histogram of Normalized {numeric_field}")
    plt.xlabel(f"{numeric_field} (z-score)")
    plt.tight_layout()
    plt.show()

# If grouping field found, show boxplot of the numeric field per group field (top 10 categories)
if group_field:
    group_counts = df[group_field].value_counts().head(10).index
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field], order=group_counts)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

* We loaded the FAIR² human protein mass spectrometry dataset using its Croissant metadata and explored its record sets by their `@id`.
* Main tables were extracted and inspected (columns, records, fields).
* A primary numeric field (e.g., molecular weight or coverage) was analyzed for distribution, outlier filtering, and normalization.
* Data was grouped on key attributes (e.g., sample, accession) when available, and visualized with histograms and boxplots.

This demonstrates how FAIR and interoperable Croissant-packaged data can be interactively analyzed with Python and `mlcroissant` by referencing all entities by their `@id` for reproducibility and clarity.